# Notebook 04 — Індекс транспортної доступності

Обчислюємо кількісний індекс доступності для кожної лікарні та школи.

## Джерела даних
- **n_stops** — OSM-зупинки + станції метро в зоні пішої доступності (з notebook 03)
- **n_routes** — унікальні маршрути через OSM-зупинки (route_ids з notebook 01)
- **frequency** — середня частота маршрутів у піковий час (з GTFS-розкладу)

## Формула

```
score = 0.4 × norm(n_stops) + 0.3 × norm(n_routes) + 0.3 × norm(frequency)
```

Нормалізація: min-max у діапазон [0, 1].

## Логіка інтеграції GTFS + OSM
OSM-зупинки мають точне розташування і вже прив'язані до маршрутів (route_ids) через
просторове з'єднання з GTFS-формами маршрутів у notebook 01.
GTFS використовується лише для розрахунку частоти рейсів — ця інформація є лише в розкладі.

In [19]:
from config_loader import cfg, ACC_RADIUS, W_STOPS, W_ROUTES, W_FREQ, WALK_10MIN_M, WALK_30MIN_M
import pandas as pd
import geopandas as gpd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='whitegrid')

# Радіус пішої доступності залежно від конфігу
WALK_RADIUS_M = WALK_10MIN_M if ACC_RADIUS == '10min' else WALK_30MIN_M
CRS_METRIC = cfg['city']['crs_metric']

print(f'Радіус ізохрони: {WALK_RADIUS_M} м ({ACC_RADIUS})')
print('Бібліотеки завантажено')

In [20]:
GTFS_PATH = cfg['paths']['gtfs']
gpkg_data_path = '../data/osm/osm_data.gpkg'

# Ізохронні результати (n_stops вже з OSM — рахувалось у notebook 03)
isochrone_df = pd.read_parquet('../data/processed/isochrone_results.parquet')

# GTFS дані (тільки для розрахунку частоти по маршрутах)
gtfs_stop_times = pd.read_parquet('../data/processed/stop_times.parquet')
gtfs_trips      = pd.read_parquet('../data/processed/trips.parquet')

gtfs: dict[str, pd.DataFrame] = {
    'stop_times': gtfs_stop_times,
    'trips':      gtfs_trips,
}

# OSM дані
osm_gstops = gpd.read_file(gpkg_data_path, layer='stops')
osm_gstops['route_ids'] = osm_gstops['route_ids'].apply(json.loads)

osm: dict[str, gpd.GeoDataFrame] = {
    'gstops': osm_gstops,
}

# Аліаси для сумісності з клітинками нижче
stop_times    = gtfs['stop_times']
trips         = gtfs['trips']
osm_stops_gdf = osm['gstops']

print(f'Об\'єктів соцінфраструктури: {len(isochrone_df)}')
print(f'  лікарні: {(isochrone_df["facility_type"]=="hospital").sum()}')
print(f'  школи:   {(isochrone_df["facility_type"]=="school").sum()}')
print(f'OSM-зупинок з route_ids: {(osm_stops_gdf["route_ids"].apply(len) > 0).sum()} з {len(osm_stops_gdf)}')

## 1. Частота маршрутів у двох часових вікнах (GTFS)

Розраховуємо середню кількість рейсів на годину для кожного маршруту у двох вікнах:
- **Пікові години** — 07:00–09:00 та 17:00–19:00 (разом 4 год)
- **Міжпікові години** — 10:00–17:00 (7 год, будні дні; до початку вечірнього піку)

Потім, знаючи які маршрути проходять через OSM-зупинки в ізохроні, отримуємо частоту для кожного об'єкта.

In [21]:
# Пікові та міжпікові години з конфігу
_ph = cfg['peak_hours']
_op = cfg['offpeak_hours']
_t = lambda s: int(s.split(':')[0])*3600 + int(s.split(':')[1])*60

PEAK_HOURS = [(_t(_ph['morning_start']), _t(_ph['morning_end'])),
              (_t(_ph['evening_start']), _t(_ph['evening_end']))]
PEAK_DURATION_H    = _ph['total_peak_hours']       # 4 год
OFFPEAK_START      = _t(_op['start'])               # 10:00
OFFPEAK_END        = _t(_op['end'])                 # 17:00
OFFPEAK_DURATION_H = _op['total_offpeak_hours']     # 7 год

# trip_id → route_id (спільний для обох вікон)
trip_to_route = trips.set_index('trip_id')['route_id'].to_dict()

# ── ПІКОВІ ГОДИНИ ────────────────────────────────────────────────────────────
peak_st = stop_times[
    (stop_times['arrival_sec'].between(PEAK_HOURS[0][0], PEAK_HOURS[0][1])) |
    (stop_times['arrival_sec'].between(PEAK_HOURS[1][0], PEAK_HOURS[1][1]))
]
peak_trips_unique = peak_st[['trip_id']].drop_duplicates().copy()
peak_trips_unique['route_id'] = peak_trips_unique['trip_id'].map(trip_to_route)
route_peak_trips = (peak_trips_unique.dropna(subset=['route_id'])
                    .groupby('route_id')['trip_id'].count())
display(route_peak_trips.head(3))
route_frequency_peak = (route_peak_trips / PEAK_DURATION_H).to_dict()

display(route_frequency_peak.head(3))
# ── МІЖПІКОВІ ГОДИНИ ─────────────────────────────────────────────────────────
offpeak_st = stop_times[
    stop_times['arrival_sec'].between(OFFPEAK_START, OFFPEAK_END)
]
offpeak_trips_unique = offpeak_st[['trip_id']].drop_duplicates().copy()
offpeak_trips_unique['route_id'] = offpeak_trips_unique['trip_id'].map(trip_to_route)
route_offpeak_trips = (offpeak_trips_unique.dropna(subset=['route_id'])
                       .groupby('route_id')['trip_id'].count())
route_frequency_offpeak = (route_offpeak_trips / OFFPEAK_DURATION_H).to_dict()

peak_vals    = list(route_frequency_peak.values())
offpeak_vals = list(route_frequency_offpeak.values())
print(f'Маршрутів (пік):             {len(route_frequency_peak)}')
print(f'Маршрутів (міжпік):          {len(route_frequency_offpeak)}')
print(f'Середня частота пік:         {np.mean(peak_vals):.1f} рейс/год')
print(f'Середня частота міжпік:      {np.mean(offpeak_vals):.1f} рейс/год')

## 2. OSM-зупинки в зоні доступності кожного об'єкта

Для кожної лікарні/школи знаходимо всі OSM-зупинки в радіусі `WALK_RADIUS_M` метрів
та збираємо унікальні route_ids — маршрути що реально проходять через ці зупинки.

In [18]:
# Проєктуємо об'єкти та OSM-зупинки у метричну CRS для точних відстаней
facilities_gdf = gpd.GeoDataFrame(
    isochrone_df[['facility_id']],
    geometry=gpd.points_from_xy(isochrone_df['lon'], isochrone_df['lat']),
    crs='EPSG:4326'
).to_crs(CRS_METRIC)

osm_proj = osm_stops_gdf[['osm_stop_id', 'route_ids', 'geometry']].to_crs(CRS_METRIC).copy()

# Буфер WALK_RADIUS_M навколо кожного об'єкта
facilities_buf = facilities_gdf.copy()
facilities_buf['geometry'] = facilities_gdf.geometry.buffer(WALK_RADIUS_M)

# Просторове з'єднання: які OSM-зупинки потрапляють у буфер?
joined = gpd.sjoin(osm_proj, facilities_buf, how='inner', predicate='within')
joined.head(3)
# Для кожного об'єкта: збираємо всі унікальні route_ids зупинок у зоні
def collect_routes(group):
    all_routes = set()
    for route_list in group['route_ids']:
        all_routes.update(r for r in route_list if r)
    return list(all_routes)

facility_routes = (
    joined.groupby('facility_id')
    .apply(collect_routes, include_groups=False)
    .reset_index()
)
facility_routes.columns = ['facility_id', 'osm_route_ids']

isochrone_df = isochrone_df.merge(facility_routes, on='facility_id', how='left')
isochrone_df['osm_route_ids'] = isochrone_df['osm_route_ids'].apply(
    lambda x: x if isinstance(x, list) else []
)

n_with_routes = (isochrone_df['osm_route_ids'].apply(len) > 0).sum()
print(f'Об\'єктів з маршрутами (OSM): {n_with_routes} з {len(isochrone_df)}')
print(f'Середня к-сть маршрутів на об\'єкт: {isochrone_df["osm_route_ids"].apply(len).mean():.1f}')

## 3. Обчислення компонент індексу

In [5]:
def compute_accessibility_components(osm_route_ids, freq_peak_dict, freq_offpeak_dict):
    """
    Обчислює n_routes і частоту для обох часових вікон.

    Параметри:
        osm_route_ids    — список route_id маршрутів через OSM-зупинки в зоні
        freq_peak_dict   — route_id → рейсів/год у піковий час (з GTFS)
        freq_offpeak_dict — route_id → рейсів/год у міжпіковий час (з GTFS)

    Повертає:
        n_routes, freq_peak, freq_offpeak
    """
    n_routes = len(osm_route_ids)

    freqs_peak = [freq_peak_dict[r] for r in osm_route_ids if r in freq_peak_dict]
    freq_peak  = float(np.mean(freqs_peak)) if freqs_peak else 0.0

    freqs_offpeak = [freq_offpeak_dict[r] for r in osm_route_ids if r in freq_offpeak_dict]
    freq_offpeak  = float(np.mean(freqs_offpeak)) if freqs_offpeak else 0.0

    return n_routes, freq_peak, freq_offpeak


results_components = isochrone_df['osm_route_ids'].apply(
    lambda r: compute_accessibility_components(r, route_frequency_peak, route_frequency_offpeak)
)

# n_stops — з OSM (рахувалось у notebook 03, ізохрона по вуличній мережі)
isochrone_df['n_stops']           = isochrone_df[f'n_stops_{ACC_RADIUS}']
isochrone_df['n_routes']          = results_components.apply(lambda x: x[0])
isochrone_df['frequency_peak']    = results_components.apply(lambda x: x[1])
isochrone_df['frequency_offpeak'] = results_components.apply(lambda x: x[2])

# Аліас для сумісності
isochrone_df['frequency'] = isochrone_df['frequency_peak']

print('=== Компоненти індексу (OSM route_ids + GTFS) ===')
print(f'  Середнє n_stops:                      {isochrone_df["n_stops"].mean():.1f}')
print(f'  Середнє n_routes:                     {isochrone_df["n_routes"].mean():.1f}')
print(f'  Середня frequency_peak (рейс/год):    {isochrone_df["frequency_peak"].mean():.2f}')
print(f'  Середня frequency_offpeak (рейс/год): {isochrone_df["frequency_offpeak"].mean():.2f}')
print(f'  Об\'єктів з 0 маршрутами:              {(isochrone_df["n_routes"] == 0).sum()}')

## 4. Нормалізація та два фінальні індекси

MIN-MAX нормалізація: `norm(x) = (x - min) / (max - min)`

Ваги однакові для обох вікон: 0.4 (зупинки) + 0.3 (маршрути) + 0.3 (частота) = 1.0

- **score_peak** — індекс у пікові години (07-09, 17-19)
- **score_offpeak** — індекс у міжпіковий час (10-17)
- **score_delta** = score_peak − score_offpeak (деградація поза піком; > 0 означає погіршення)

In [6]:
def minmax_normalize(series):
    """Min-max нормалізація в [0, 1]."""
    min_val, max_val = series.min(), series.max()
    if max_val == min_val:
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - min_val) / (max_val - min_val)


# Спільні компоненти (не залежать від часового вікна)
norm_stops  = minmax_normalize(isochrone_df['n_stops'])
norm_routes = minmax_normalize(isochrone_df['n_routes'])

# Частота нормалізується окремо для кожного вікна
norm_freq_peak    = minmax_normalize(isochrone_df['frequency_peak'])
norm_freq_offpeak = minmax_normalize(isochrone_df['frequency_offpeak'])

isochrone_df['norm_stops']        = norm_stops
isochrone_df['norm_routes']       = norm_routes
isochrone_df['norm_freq_peak']    = norm_freq_peak
isochrone_df['norm_freq_offpeak'] = norm_freq_offpeak
isochrone_df['norm_frequency']    = norm_freq_peak   # аліас для сумісності

# Два індекси (формула однакова, відрізняється лише компонента частоти)
isochrone_df['score_peak'] = (
    W_STOPS  * norm_stops +
    W_ROUTES * norm_routes +
    W_FREQ   * norm_freq_peak
)
isochrone_df['score_offpeak'] = (
    W_STOPS  * norm_stops +
    W_ROUTES * norm_routes +
    W_FREQ   * norm_freq_offpeak
)
isochrone_df['score_delta'] = isochrone_df['score_peak'] - isochrone_df['score_offpeak']

# accessibility_score = score_peak — для сумісності з NB05/NB06/NB07
isochrone_df['accessibility_score'] = isochrone_df['score_peak']

print('=== Два індекси доступності ===')
for col, label in [
    ('score_peak',    'Пік (07-09, 17-19)       '),
    ('score_offpeak', 'Міжпік (10-16)            '),
    ('score_delta',   'Деградація (пік − міжпік)'),
]:
    s = isochrone_df[col]
    print(f'  {label}: mid={s.mean():.3f}, min={s.min():.3f}, max={s.max():.3f}')

n_degraded = (isochrone_df['score_delta'] > 0.05).sum()
print(f'\n  Об\'єктів з суттєвою деградацією (Δ > 0.05): {n_degraded} ({100*n_degraded/len(isochrone_df):.1f}%)')

In [7]:
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# ── 1. KDE: розподіл score_peak vs score_offpeak по типах об'єктів ────────────
for col, color, label in [
    ('score_peak',    '#2196F3', 'Пік (07-09, 17-19)'),
    ('score_offpeak', '#FF9800', 'Міжпік (10-17)'),
]:
    for ftype, ls in [('hospital', '-'), ('school', '--')]:
        subset = isochrone_df[isochrone_df['facility_type'] == ftype][col]
        subset.plot.kde(ax=axes[0, 0], color=color, linestyle=ls, linewidth=2,
                        label=f'{label} — {"лікарні" if ftype == "hospital" else "школи"}')
axes[0, 0].set_title('Розподіл індексу: пік vs міжпік', fontsize=12)
axes[0, 0].set_xlabel('Індекс доступності')
axes[0, 0].legend(fontsize=8)
axes[0, 0].set_xlim(0, 1)

# ── 2. Scatter: score_peak vs score_offpeak (кожен об'єкт) ───────────────────
for ftype, color, marker in [('hospital', '#F44336', 'o'), ('school', '#4CAF50', '^')]:
    sub = isochrone_df[isochrone_df['facility_type'] == ftype]
    axes[0, 1].scatter(sub['score_peak'], sub['score_offpeak'],
                       c=color, marker=marker, alpha=0.5, s=25, edgecolors='none',
                       label='Лікарні' if ftype == 'hospital' else 'Школи')
axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Рівність (пік = міжпік)')
axes[0, 1].set_xlabel('score_peak')
axes[0, 1].set_ylabel('score_offpeak')
axes[0, 1].set_title('score_peak vs score_offpeak (кожен об\'єкт)', fontsize=12)
axes[0, 1].legend()

# ── 3. Гістограма деградації (score_delta) ────────────────────────────────────
axes[1, 0].hist(isochrone_df['score_delta'], bins=30, color='#9C27B0',
                edgecolor='white', alpha=0.85)
axes[1, 0].axvline(0, color='black', linestyle='--', linewidth=1.5, label='Нема деградації')
mean_d = isochrone_df['score_delta'].mean()
axes[1, 0].axvline(mean_d, color='red', linewidth=1.5,
                   label=f'Середнє Δ = {mean_d:.3f}')
axes[1, 0].set_title('Деградація доступності поза піком (score_delta)', fontsize=12)
axes[1, 0].set_xlabel('score_peak − score_offpeak')
axes[1, 0].set_ylabel('Кількість об\'єктів')
axes[1, 0].legend()

# ── 4. Box plot: лікарні vs школи × пік vs міжпік ────────────────────────────
data_box, labels_box = [], []
for ftype, ftname in [('hospital', 'Лікарні'), ('school', 'Школи')]:
    sub = isochrone_df[isochrone_df['facility_type'] == ftype]
    data_box += [sub['score_peak'].values, sub['score_offpeak'].values]
    labels_box += [f'{ftname}\nПік', f'{ftname}\nМіжпік']

bp = axes[1, 1].boxplot(data_box, labels=labels_box, patch_artist=True,
                         medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], ['#2196F3', '#FF9800', '#2196F3', '#FF9800']):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
axes[1, 1].set_title('Box plot: пік vs міжпік по типах об\'єктів', fontsize=12)
axes[1, 1].set_ylabel('Індекс доступності')
peak_patch    = mpatches.Patch(color='#2196F3', alpha=0.75, label='Пік (07-09, 17-19)')
offpeak_patch = mpatches.Patch(color='#FF9800', alpha=0.75, label='Міжпік (10-17)')
axes[1, 1].legend(handles=[peak_patch, offpeak_patch])

plt.suptitle('Порівняльний аналіз: пікова vs міжпікова доступність', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/peak_vs_offpeak_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Збережено: outputs/peak_vs_offpeak_comparison.png')

In [8]:
cols_to_save = [
    'facility_id', 'facility_type', 'name', 'lon', 'lat',
    'n_stops', 'n_routes',
    'frequency_peak', 'frequency_offpeak',
    'norm_stops', 'norm_routes', 'norm_freq_peak', 'norm_freq_offpeak',
    'score_peak', 'score_offpeak', 'score_delta',
    'accessibility_score',   # = score_peak, для сумісності
    'frequency',             # = frequency_peak, для сумісності
    'norm_frequency',        # = norm_freq_peak, для сумісності
]
accessibility_df = isochrone_df[cols_to_save].copy()
accessibility_df.to_csv('../data/processed/accessibility_scores.csv', index=False, encoding='utf-8')

print('Файл збережено: data/processed/accessibility_scores.csv')
print(f'Рядків: {len(accessibility_df)}')
print()
print('=== Топ-5 за score_peak ===')
accessibility_df.sort_values('score_peak', ascending=False)[
    ['name', 'facility_type', 'score_peak', 'score_offpeak', 'score_delta']
].head()

In [9]:
# Транспортні "мертві зони" — найнижчі індекси
print('=== ТРАНСПОРТНІ МЕРТВІ ЗОНИ (score_peak = 0..0.15) ===')
bottom = accessibility_df.nsmallest(10, 'score_peak')
print(bottom[['name', 'facility_type', 'n_stops', 'n_routes',
              'frequency_peak', 'frequency_offpeak',
              'score_peak', 'score_offpeak', 'score_delta']].to_string(index=False))
print()
print('Notebook 04 виконано. Переходьте до 05_clustering.ipynb')